# Begin

Courtesy: https://docs.cleanrl.dev/rl-algorithms/ppo-trxl/

In [1]:
# @launchit.collected

In [2]:
import os # @launchit.collect
import sys # @launchit.collect
import copy
from collections import namedtuple, defaultdict, Counter, deque # @launchit.collect
import random
import datetime
import json
import pprint
import re
import uuid
from unittest.mock import Mock
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import IPython
from enum import Flag, StrEnum, auto # @launchit.collect
import multiprocessing as mp

import lark # @launchit.collect

from tqdm.notebook import tqdm

import numpy as np
import cupy as cp
import einops
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch.utils.data import Dataset, DataLoader
from torch.distributions import Categorical

import gymnasium as gym
import memory_gym 
import av

import optuna 
from optuna.storages import JournalStorage 
from optuna.storages.journal import JournalFileBackend 
from optuna.trial import TrialState

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect

import lang_utils as lu # @launchit.collect
import array_utils as au # @launchit.collect
from math_utils import RecursiveAverageFilter
from logging_utils import *
from artifact_registry import *
from torch_utils import *
import launchit # @launchit.disable
import optuna_multiprocessing  # @launchit.collect
from hp_utils import *
from metrics_collector import RmqSummaryWriter
from autoincrement import Autoincrement

/home/misha/anaconda3/envs/mine/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


# Init

In [3]:
class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()
    
def create_config():
    config = namedtuple('Config', 
                        'project_root_path, project_root_uri, model_group_uri, subproject_path, data_path, private_data_path, run_path, ' + 
                        'self_fname, self_name, ' +
                        'subproject_name,' +
                        'is_cuda, cuda_device, exec_mode, is_interactive')(
        project_root_path=project_root_path,
        project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
        model_group_uri=None,
        subproject_path=os.path.abspath('.'),
        data_path=os.path.join(project_root_path, 'data'),
        private_data_path=None,
        run_path=None,
        self_fname=None,
        self_name=None,
        subproject_name=None,
        is_cuda=torch.cuda.is_available(),
        cuda_device='cuda' if torch.cuda.is_available() else 'cpu',
        exec_mode=ExecMode.MASTER_NOTEBOOK,
        is_interactive=True,
    )
    
    if IPython.get_ipython() is None:
        module_fname = __file__
        module_basename = os.path.basename(module_fname)
        module_name, _ = os.path.splitext(module_basename)
        
        config = config._replace(self_fname=module_fname, self_name=module_name)
        config = config._replace(exec_mode=ExecMode.LAUNCH_MODULE)
    else:
        with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
            notebook_fname = json.load(cf)['jupyter_session']
            notebook_basename = os.path.basename(notebook_fname)
            notebook_name, notebook_ext = os.path.splitext(notebook_basename)
        
            m = re.match(r'(\w+)-Copy\d+$', notebook_name)
        
            if m: notebook_name = m.group(1) # e.g. Cuml is used to be launched from the copy of the notebook
    
            config = config._replace(self_fname=notebook_fname, self_name=notebook_name)
            
            is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None
            config = config._replace(exec_mode=ExecMode.MASTER_NOTEBOOK if not is_launch else ExecMode.LAUNCH_NOTEBOOK)
    
    config = config._replace(is_interactive=config.exec_mode != ExecMode.LAUNCH_MODULE)    
    config = config._replace(subproject_name=os.path.basename(os.path.dirname(config.self_fname)))
    config = config._replace(model_group_uri=f'{config.project_root_uri}.{config.subproject_name}')
    config = config._replace(run_path=os.path.join(project_root_path, 'run', config.subproject_name))
    config = config._replace(private_data_path=os.path.join(config.data_path, config.subproject_name))
    return config

In [4]:
au.init()
LOG = Logging.get()
RNG = np.random.default_rng()
METRICS_SUITE = defaultdict(list)
CONFIG = create_config()
LOG.app_name = CONFIG.self_name
LOG.enable('syslog', not CONFIG.is_interactive)
LOG.enable('stdout', CONFIG.is_interactive)
LOG(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n', when=CONFIG.is_interactive)
LOG(f'CONFIG={CONFIG._asdict()}', when=not CONFIG.is_interactive)
os.makedirs(CONFIG.private_data_path, exist_ok=True)
os.makedirs(CONFIG.run_path, exist_ok=True)

CONFIG=
{'project_root_path': '/home/misha/dev/mine/neurolab',
 'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.17_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/17_rl',
 'data_path': '/home/misha/dev/mine/neurolab/data',
 'private_data_path': '/home/misha/dev/mine/neurolab/data/17_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/17_rl',
 'self_fname': '/home/misha/dev/mine/neurolab/17_rl/17c_ppo_trxl_memory_01.ipynb',
 'self_name': '17c_ppo_trxl_memory_01',
 'subproject_name': '17_rl',
 'is_cuda': True,
 'cuda_device': 'cuda',
 'exec_mode': <ExecMode.MASTER_NOTEBOOK: 'master_notebook'>,
 'is_interactive': True}



# Hyperparameters

In [5]:
# @launchit.disable
# @launchit.collect
class LaunchGoal(StrEnum):
    UNSPECIFIED = auto()
    TRAIN = auto()

LaunchComponent = namedtuple('LaunchComponent', 'name version uri main_asset_fname')
    
@dataclass(slots=True)
class Hyperparameters:
    # Launch
    launch_goal: LaunchGoal = lu.from_str(LaunchGoal, '${LAUNCH_GOAL}', LaunchGoal.UNSPECIFIED)
    launch_id: int = lu.from_str(int, '${MODEL_VERSION}', 0)

    # System params
    random_seed: int = None
    torch_deterministic: bool = True

    # Environment params
    env_id: str = None
    envs_count: int = 1 # the number of parallel game environments

    # Transformer-XL Agent params
    trxl_layers_count: int = 3 # number of transformer layers
    trxl_heads_count: int = 4 # number of heads used in multi-head attention
    trxl_d_model: int = 384 # the dimension of the transformer
    trxl_memory_length: int = 119 # the length of TrXL's sliding memory window
    trxl_positional_encoding: str = "absolute" # positional encoding type of the transformer: "", "absolute", "learned"
    
    # RL params
    gamma: float = 0.995 # return discount factor gamma
    gae_lambda: float = 0.95 # lambda for the general advantage estimation
    
    # Training procedure params (PPO related) 
    global_steps_count: int = 1_000_000 # total number of steps 
    rollout_steps_count: int = 512 # how many steps to run in a single policy rolllout
    anneal_steps_count: int = 1 * 512 * 10_000 # anneal steps count for learn rate and entropy coeff
    minibatches_count: int = 8
    epochs_count: int = 3 
    clip_coef: float = 0.1 # the surrogate clipping coefficient
    clip_vloss: bool = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
    init_ent_coef: float = 0.0001 # initial coefficient of the entropy
    final_ent_coef: float = 0.000001 # final coefficient of the entropy
    vf_coef: float = 0.5 # coefficient of the value function
    max_grad_norm: float = 0.25 # the maximum norm for the gradient clipping
    target_kl: float = None # e target KL divergence threshold
    norm_adv: bool = False # Toggles advantages normalization
    reconstruction_coef: float = 0.0 # the coefficient of the observation reconstruction loss, if set to 0.0 the reconstruction loss is not used

    # Video params
    capture_video: str = 'every(1000000)' # video capture policy depending on steps

    # Optimization params
    optimizer: str = 'Adam'
    init_learn_rate: float = 2.75e-4
    final_learn_rate: float = 1.0e-5

    @staticmethod
    def from_dict(d):
        hp = Hyperparameters(**d)
        return hp

    def _asdict(self):
        return dataclasses.asdict(self)

    def launch_component(self):
        name = lu.when('${MODEL_NAME}' == '$' + '{MODEL_NAME}', CONFIG.self_name, '${MODEL_NAME}')
        return LaunchComponent(name=name, version=self.launch_id,  uri=f'{CONFIG.model_group_uri}.{name}', main_asset_fname=CONFIG.self_fname)

HP = Hyperparameters()
HP.random_seed = 42

# Launch

## LaunchState

In [16]:
@dataclass(slots=True)
class LaunchState:
    envs: object = None
    env_observation_space: object = None
    env_action_space: object = None
    agent: object = None

    def new_artifact_registry(is_real=None):
        is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
        is_real = lu.coalesce(is_real, is_launch)
    
        if not is_real:
            mr = Mock()
            mr.register_model.return_value = 0
            return mr
            
        return ArtifactRegistry(CONFIG.model_group_uri)

    def new_summary_writer(log_dir, is_real=None):
        is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
        is_real = lu.coalesce(is_real, is_launch)
    
        if not is_real:
            sw = Mock()
            sw.flush.side_effect = sw.reset_mock # to get rid of all recorded call_args_list, which might be heavy (e.g. add_figure)
            return sw
        
        return RmqSummaryWriter(log_dir)

## Create

In [19]:
optuna_trial = optuna_multiprocessing.get_trial()
optuna_trial_subdir_name = ''

if optuna_trial is not None:
    optuna_trial.set_user_attr('MODEL_VERSION', HP.launch_id)
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    optuna_trial_subdir_name = f'opt_{study_serial}'
    LOG(f'Optuna {optuna_trial.number=}, {optuna_trial.user_attrs=}')

LOG(f'HP={HP._asdict()}', when=not CONFIG.is_interactive)
    
if HP.random_seed is not None:
    random.seed(HP.random_seed)
    torch.manual_seed(HP.random_seed)
    RNG = np.random.default_rng(HP.random_seed)    
    LOG(f'Random seed={HP.random_seed}')

if HP.torch_deterministic is not None:
    torch.backends.cudnn.deterministic = HP.torch_deterministic
    LOG(f'{torch.backends.cudnn.deterministic=}')

lc = HP.launch_component()
artifact_registry = new_artifact_registry()
artifact_registry.attach_asset(lc.name, lc.version, lc.main_asset_fname, replace=True)
    
meta = dict(
    optuna_trial_number=getattr(optuna_trial, 'number', None),
    hypers=HP._asdict(), 
    config=CONFIG._asdict(), 
)

with io.StringIO() as b:
    json.dump(meta, b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='meta', replace=True)

summary_log_dir = lc.name
summary_log_dir = os.path.join(summary_log_dir, optuna_trial_subdir_name) if optuna_trial_subdir_name != '' else summary_log_dir 
summary_log_dir = os.path.join(summary_log_dir, str(lc.version))
LOG(f'Tensorboard run={summary_log_dir}')
summary_writer = new_summary_writer(log_dir=summary_log_dir)
summary_writer.add_text('hypers', pprint.pformat(HP._asdict(), sort_dicts=False), 1)
summary_writer.add_text('config', pprint.pformat(CONFIG._asdict(), sort_dicts=False), 1)

LS = LaunchState()

Random seed=42
torch.backends.cudnn.deterministic=True
Tensorboard run=17c_ppo_trxl_memory_01/0


# Environment

## create_env

In [20]:
def create_env(env_id, video_dir_name=None, random_seed=None, is_auto_reset=False):
    env = gym.make(env_id, render_mode='debug_rgb_array')
    
    if video_dir_name is not None:
        env = gym.wrappers.RecordVideo(env, video_dir_name, episode_trigger=lambda episode_id: episode_id == 0)
        
    env = gym.wrappers.RecordEpisodeStatistics(env)

    if is_auto_reset:
        env = gym.wrappers.Autoreset(env)
    
    if random_seed is not None:
        env.action_space.seed(random_seed) # req-d for random sampling from action space when there are multiple envs

    return env

## Configure 

In [21]:
# @launchit.disable
# @launchit.collect
HP.env_id = "MortarMayhem-Grid-v0"
HP.envs_count = 1 # the number of parallel game environments
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': True,
 'env_id': 'MortarMayhem-Grid-v0',
 'envs_count': 1,
 'trxl_layers_count': 3,
 'trxl_heads_count': 4,
 'trxl_d_model': 384,
 'trxl_memory_length': 119,
 'trxl_positional_encoding': 'absolute',
 'gamma': 0.995,
 'gae_lambda': 0.95,
 'global_steps_count': 1000000,
 'rollout_steps_count': 512,
 'anneal_steps_count': 5120000,
 'minibatches_count': 8,
 'epochs_count': 3,
 'clip_coef': 0.1,
 'clip_vloss': True,
 'init_ent_coef': 0.0001,
 'final_ent_coef': 1e-06,
 'vf_coef': 0.5,
 'max_grad_norm': 0.25,
 'target_kl': None,
 'norm_adv': False,
 'reconstruction_coef': 0.0,
 'capture_video': 'every(1000000)',
 'optimizer': 'Adam',
 'init_learn_rate': 0.000275,
 'final_learn_rate': 1e-05}


## Create

In [24]:
env_factory = lambda env_id, random_seed: lambda: create_env(env_id, random_seed=random_seed)
LS.envs = gym.vector.SyncVectorEnv(
    [env_factory(HP.env_id, HP.random_seed + i) for i in range(HP.envs_count)],
    autoreset_mode=gym.vector.vector_env.AutoresetMode.NEXT_STEP, # https://farama.org/Vector-Autoreset-Mode
)
LS.env_observation_space = LS.envs.single_observation_space
LS.env_action_space = (
    (LS.envs.single_action_space.n,)
    if isinstance(LS.envs.single_action_space, gym.spaces.Discrete)
    else tuple(envs.single_action_space.nvec)
)
LOG(f'{LS.env_observation_space=}')
LOG(f'{LS.env_action_space=}')
LOG(f'{LS.envs.metadata=}')

LS.env_observation_space=Box(0, 255, (84, 84, 3), uint8)
LS.env_action_space=(np.int64(4),)
LS.envs.metadata={'render_modes': ['human', 'rgb_array', 'debug_rgb_array'], 'render_fps': 6, 'autoreset_mode': <AutoresetMode.NEXT_STEP: 'NextStep'>}


ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5727:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM default


# Agent

## Components

### layer_init

In [128]:
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    # torch.nn.init.constant_(layer.bias, bias_const)
    return layer

### PositionalEncoding

In [122]:
# dialogs/positional_embedding.ipynb
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, min_timescale=2.0, max_timescale=1e4):
        super().__init__()
        freqs = torch.arange(0, d_model, min_timescale) # e.g. -> [0, 2, 4, ..., 382], len(freqs) = d_model // min_timescale
        inv_freqs = max_timescale ** (-freqs / d_model) # e.g. [1, 0.96, 0.93, ... 0.001]
        self.register_buffer("inv_freqs", inv_freqs) # declare as non trainable parameter but still part of a model (e.g. included in state_dict, obeys to(device), etc.)

    def forward(self, seq_len):
        seq = torch.arange(seq_len - 1, -1, -1.0) # -> [seq_len-1, seq_len-2, ... 0]
        sinusoidal_inp = einops.rearrange(seq, 'n -> n 1') * einops.rearrange(self.inv_freqs, 'd -> 1 d') # n -> n () == n -> n 1
        pos_emb = torch.cat((sinusoidal_inp.sin(), sinusoidal_inp.cos()), dim=-1)
        return pos_emb # [seq_len, d_model]

### MultiHeadAttention

In [123]:
class MultiHeadAttention(nn.Module):
    """Multi Head Attention without dropout inspired by https://github.com/aladdinpersson/Machine-Learning-Collection"""

    def __init__(self, d_model, heads_count):
        super().__init__()
        self.d_model = d_model
        self.heads_count = heads_count
        self.head_size = d_model // heads_count # aka head_dim, d_head

        assert self.head_size * heads_count == d_model, 'Embedding dimension needs to be divisible by the number of heads'

        self.values = nn.Linear(self.head_size, self.head_size, bias=False)
        self.keys = nn.Linear(self.head_size, self.head_size, bias=False)
        self.queries = nn.Linear(self.head_size, self.head_size, bias=False)
        self.fc_out = nn.Linear(self.heads_count * self.head_size, d_model)

    def forward(self, values, keys, query, mask):
        N = query.shape[0]
        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

        values = values.reshape(N, value_len, self.heads_count, self.head_size)
        keys = keys.reshape(N, key_len, self.heads_count, self.head_size)
        query = query.reshape(N, query_len, self.heads_count, self.head_size)

        values = self.values(values)  # (N, value_len, heads, head_dim)
        keys = self.keys(keys)  # (N, key_len, heads, head_dim)
        queries = self.queries(query)  # (N, query_len, heads, head_dim)

        # Dot-product
        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys]) # (N, heads, query_len, key_len)

        # Mask padded indices so their attention weights become 0
        if mask is not None:
            energy = energy.masked_fill(mask.unsqueeze(1).unsqueeze(1) == 0, float("-1e20"))  # -inf causes NaN

        # Normalize energy values and apply softmax to retrieve the attention scores
        attention = torch.softmax(
            energy / (self.d_model ** (1 / 2)), dim=3
        )  # attention shape: (N, heads, query_len, key_len)

        # Scale values by attention weights
        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]) # (N, query_len, heads, head_dim)
        out = out.reshape(N, query_len, self.heads_count * self.head_size)  # (N, query_len, d_model)

        return self.fc_out(out), attention

### TransformerLayer

In [124]:
class TransformerLayer(nn.Module):
    def __init__(self, d_model, heads_count):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, heads_count)
        self.layer_norm_q = nn.LayerNorm(d_model)
        self.norm_kv = nn.LayerNorm(d_model)
        self.layer_norm_attn = nn.LayerNorm(d_model)
        self.fc_projection = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU())

    def forward(self, value, key, query, mask):
        # Pre-layer normalization (post-layer normalization is usually less effective)
        query_ = self.layer_norm_q(query)
        value = self.norm_kv(value)
        key = value  # K = V -> self-attention
        attention, attention_weights = self.attention(value, key, query_, mask)  # MHA
        x = attention + query  # Skip connection
        x_ = self.layer_norm_attn(x)  # Pre-layer normalization
        forward = self.fc_projection(x_)  # Forward projection
        out = forward + x  # Skip connection
        return out, attention_weights

### Transformer

In [165]:
class Transformer(nn.Module):
    def __init__(self, layers_count, d_model, heads_count, max_episode_steps, positional_encoding):
        super().__init__()
        self.max_episode_steps = max_episode_steps
        self.positional_encoding = positional_encoding
        
        assert positional_encoding in ['absolute', 'learned'], f'Unsupported {positional_encoding=}'
        
        if positional_encoding == 'absolute':
            self.pos_embedding = PositionalEncoding(d_model)
        elif positional_encoding == 'learned':
            self.pos_embedding = nn.Parameter(torch.randn(max_episode_steps, d_model))
            
        self.transformer_layers = nn.ModuleList([TransformerLayer(d_model, heads_count) for _ in range(layers_count)])

    def forward(self, x, memories, mask, memory_indices):
        # Add positional encoding to every transformer layer input
        if self.positional_encoding == 'absolute':
            pos_embedding = self.pos_embedding(self.max_episode_steps)[memory_indices] # kms@ everytime computation of sin/cos for 1000*384 elements?!
            print(f'kms@ {memories.shape=} {pos_embedding.shape=} {pos_embedding.unsqueeze(2).shape=}')
            memories = memories + pos_embedding.unsqueeze(2)
        elif self.positional_encoding == 'learned':
            memories = memories + self.pos_embedding[memory_indices].unsqueeze(2)

        # Forward transformer layers and return new memories (i.e. hidden states)
        out_memories = []
        
        for i, layer in enumerate(self.transformer_layers):
            out_memories.append(x.detach()) # kms@ Stop-Gradient?
            x, attention_weights = layer(
                value=memories[:, :, i], 
                key=memories[:, :, i], 
                query=x.unsqueeze(1),
                mask=mask
            ) 
            x = x.squeeze()
            
            if len(x.shape) == 1:
                x = x.unsqueeze(0)
        
        return x, torch.stack(out_memories, dim=1)

## Agent

In [225]:
class Agent(nn.Module):
    @dataclass(slots=True)
    class Params:
        d_model: int = None
        layers_count: int = None
        heads_count: int = None
        positional_encoding: str = None
        observation_space_shape: tuple = None
        action_space_shape: tuple = None
        max_episode_steps: int = None
        with_reconstruction_head: bool = None
        
    def __init__(self, params):
        super().__init__()
        self.params = params

        if len(self.params.observation_space_shape) > 1:
            self.encoder = nn.Sequential(
                layer_init(nn.Conv2d(3, 32, 8, stride=4)),
                nn.ReLU(),
                layer_init(nn.Conv2d(32, 64, 4, stride=2)),
                nn.ReLU(),
                layer_init(nn.Conv2d(64, 64, 3, stride=1)),
                nn.ReLU(),
                nn.Flatten(),
                layer_init(nn.Linear(64 * 7 * 7, self.params.d_model)), # kms@ collapse 64 features maps of 7x7 grid (49 elements) to just single vector in embedding space
                nn.ReLU(),
            )
        else:
            self.encoder = layer_init(nn.Linear(observation_space.shape[0], self.params.d_model))

        self.transformer = Transformer(
            self.params.layers_count, 
            self.params.d_model, 
            self.params.heads_count, 
            self.params.max_episode_steps, 
            self.params.positional_encoding,
        )

        # kms@ extra MLP layer?
        self.hidden_post_trxl = nn.Sequential(
            layer_init(nn.Linear(self.params.d_model, self.params.d_model)),
            nn.ReLU(),
        )

        # kms@ Create action vectors for multi-descrete actions
        self.actor_branches = nn.ModuleList(
            [
                layer_init(nn.Linear(self.params.d_model, out_features=actions_count), np.sqrt(0.01))
                for actions_count in self.params.action_space_shape
            ]
        )
        self.critic = layer_init(nn.Linear(self.params.d_model, 1), 1)

        if params.with_reconstruction_head:
            self.transposed_cnn = nn.Sequential(
                layer_init(nn.Linear(self.params.d_model, 64 * 7 * 7)),
                nn.ReLU(),
                nn.Unflatten(1, (64, 7, 7)),
                layer_init(nn.ConvTranspose2d(64, 64, 3, stride=1)),
                nn.ReLU(),
                layer_init(nn.ConvTranspose2d(64, 32, 4, stride=2)),
                nn.ReLU(),
                layer_init(nn.ConvTranspose2d(32, 3, 8, stride=4)),
                nn.Sigmoid(),
            )

    def get_value(self, x, memory, memory_mask, memory_indices):
        if len(self.params.observation_space_shape) > 1:
            x = self.encoder(x.permute((0, 3, 1, 2)) / 255.0) # (batch, color, height, width), planar layout
        else:
            x = self.encoder(x)
            
        x, _ = self.transformer(x, memory, memory_mask, memory_indices)
        x = self.hidden_post_trxl(x) # kms@ !!!
        return self.critic(x).flatten()

    ForwardResult = namedtuple('ForwardResult', 'actions, action_log_probs, prob_entropies, values, memories')

    def get_action_and_value(self, x, memory, memory_mask, memory_indices, action=None):
        if len(self.params.observation_space_shape) > 1:
            x = self.encoder(x.permute((0, 3, 1, 2)) / 255.0) # (batch, color, height, width), planar layout
        else:
            x = self.encoder(x)

        # Here we have an observation packed into a single token mapped to embedding space
        # print(f'kms@ {x.shape=}')
            
        x, memory = self.transformer(x, memory, memory_mask, memory_indices)
        x = self.hidden_post_trxl(x) # kms@ !!!
        
        self.x = x
        probs = [Categorical(logits=branch(x)) for branch in self.actor_branches]
        
        if action is None:
            action = torch.stack([dist.sample() for dist in probs], dim=1)
            
        log_probs = []
        
        for i, dist in enumerate(probs):
            log_probs.append(dist.log_prob(action[:, i]))
            
        entropies = torch.stack([dist.entropy() for dist in probs], dim=1).sum(1).reshape(-1)
        return Agent.ForwardResult(
            actions=action,
            action_log_probs=torch.stack(log_probs, dim=1),
            prob_entropies=entropies,
            values=self.critic(x).flatten(),
            memories=memory,
        )

    def reconstruct_observation(self):
        x = self.transposed_cnn(self.x)
        return x.permute((0, 2, 3, 1)) # (batch, height, width, color), interleaved (RGB) layout

## Smoke test

In [223]:
max_episode_steps = 300
memory_length = 100

repetitions = torch.repeat_interleave(
    torch.arange(0, memory_length).unsqueeze(0), 
    memory_length - 1, 
    dim=0
).long() # [memory_length-1, memory_length], e.g. [99, 100], where each row=[0, 1, ... 99]
memory_indices_1 = torch.stack(
    [torch.arange(i, i + memory_length) for i in range(max_episode_steps - memory_length + 1)]
).long() # [max_episode_steps - memory_length + 1, memory_length], e.g. [201, 100]
memory_indices = torch.cat((repetitions, memory_indices_1))

In [191]:
repetitions.shape

torch.Size([99, 100])

In [192]:
repetitions

tensor([[ 0,  1,  2,  ..., 97, 98, 99],
        [ 0,  1,  2,  ..., 97, 98, 99],
        [ 0,  1,  2,  ..., 97, 98, 99],
        ...,
        [ 0,  1,  2,  ..., 97, 98, 99],
        [ 0,  1,  2,  ..., 97, 98, 99],
        [ 0,  1,  2,  ..., 97, 98, 99]])

In [196]:
memory_indices_1.shape

torch.Size([201, 100])

In [197]:
memory_indices_1

tensor([[  0,   1,   2,  ...,  97,  98,  99],
        [  1,   2,   3,  ...,  98,  99, 100],
        [  2,   3,   4,  ...,  99, 100, 101],
        ...,
        [198, 199, 200,  ..., 295, 296, 297],
        [199, 200, 201,  ..., 296, 297, 298],
        [200, 201, 202,  ..., 297, 298, 299]])

In [198]:
memory_indices.shape

torch.Size([300, 100])

In [199]:
memory_indices

tensor([[  0,   1,   2,  ...,  97,  98,  99],
        [  0,   1,   2,  ...,  97,  98,  99],
        [  0,   1,   2,  ...,  97,  98,  99],
        ...,
        [198, 199, 200,  ..., 295, 296, 297],
        [199, 200, 201,  ..., 296, 297, 298],
        [200, 201, 202,  ..., 297, 298, 299]])

In [216]:
def batched_index_select(input, dim, index):
    print(f'kms@ a) {index.shape=}')
    
    for ii in range(1, len(input.shape)):
        if ii != dim:
            index = index.unsqueeze(ii)

    print(f'kms@ b) {index.shape=}')
            
    expanse = list(input.shape)
    expanse[0] = -1
    expanse[dim] = -1
    index = index.expand(expanse)

    print(f'kms@ b) {input.shape=}')
    
    return torch.gather(input, dim, index)

In [220]:
envs_count = 1
steps_count = 100

a = torch.zeros((envs_count * steps_count * ap.layers_count * ap.d_model)).reshape(envs_count, steps_count, ap.layers_count, ap.d_model)
step = torch.zeros((envs_count,), dtype=torch.long)
a2 = batched_index_select(a, 1, memory_indices[step])
a.shape, a2.shape, memory_indices[step].shape

kms@ a) index.shape=torch.Size([1, 100])
kms@ b) index.shape=torch.Size([1, 100, 1, 1])
kms@ b) input.shape=torch.Size([1, 100, 3, 384])


(torch.Size([1, 100, 3, 384]),
 torch.Size([1, 100, 3, 384]),
 torch.Size([1, 100]))

In [245]:
# @launchit.disable
device = 'cpu'
envs_count = 2
steps_count = 10

ap = Agent.Params(
    d_model=384,
    layers_count=3,
    heads_count=4,
    positional_encoding='absolute',
    observation_space_shape=(84, 84, 3),
    action_space_shape=(4,),
    max_episode_steps=1000,
    with_reconstruction_head=False,
)
agent = Agent(ap)
agent = agent.to(device)
print(agent)
params_count = sum(p.numel() for p in agent.parameters())
print(f'{params_count=:_}')
probe_batch = torch.zeros((envs_count,) + ap.observation_space_shape).to(device)
print(f'{probe_batch.shape=}')

r = agent.get_action_and_value(
    x=probe_batch, 
    memory=torch.zeros((envs_count, steps_count, ap.layers_count, ap.d_model)),
    memory_mask=None,
    memory_indices=torch.zeros((envs_count, steps_count), dtype=torch.long),
)

shape = einops.parse_shape(r.actions, 'e a')
assert shape['e'] == envs_count
assert shape['a'] == len(ap.action_space_shape)
print(f'{r.actions.shape=}, {shape=}')

shape = einops.parse_shape(r.action_log_probs, 'e a')
assert shape['e'] == envs_count
assert shape['a'] == len(ap.action_space_shape)
print(f'{r.action_log_probs.shape=}, {shape=}')

shape = einops.parse_shape(r.prob_entropies, 'e')
assert shape['e'] == envs_count
print(f'{r.prob_entropies.shape=}, {shape=}')

shape = einops.parse_shape(r.values, 'e')
assert shape['e'] == envs_count
print(f'{r.values.shape=}, {shape=}')

shape = einops.parse_shape(r.memories, 'e l d')
assert shape['e'] == envs_count
assert shape['l'] == ap.layers_count
assert shape['d'] == ap.d_model
print(f'{r.memories.shape=}, {shape=}')

Agent(
  (encoder): Sequential(
    (0): Conv2d(3, 32, kernel_size=(8, 8), stride=(4, 4))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
    (3): ReLU()
    (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (5): ReLU()
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=3136, out_features=384, bias=True)
    (8): ReLU()
  )
  (transformer): Transformer(
    (pos_embedding): PositionalEncoding()
    (transformer_layers): ModuleList(
      (0-2): 3 x TransformerLayer(
        (attention): MultiHeadAttention(
          (values): Linear(in_features=96, out_features=96, bias=False)
          (keys): Linear(in_features=96, out_features=96, bias=False)
          (queries): Linear(in_features=96, out_features=96, bias=False)
          (fc_out): Linear(in_features=384, out_features=384, bias=True)
        )
        (layer_norm_q): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (norm_kv): LayerNorm((384,), eps=1e-05, elementwi

## Quick play test

In [164]:
# @launchit.disable
ap = Agent.Params(
    actions_count=LS.envs.single_action_space.n.item(),
)
agent = Agent(ap)
agent = agent.to(CONFIG.cuda_device)
env = create_env(HP.env_id, is_auto_reset=True)
obs, _ = env.reset(seed=HP.random_seed)
device = next(iter(agent.parameters())).device

with eval_guard(agent):
    with torch.no_grad():
        for step in tqdm(range(0, 1000)): 
            obs = torch.tensor(einops.rearrange(obs, 'f h w -> 1 f h w')).to(device)
            logits = agent(obs)[0]
            action = torch.argmax(logits).cpu().item()
            obs, reward, terminated, truncated, info = env.step(action)

            if terminated or truncated:
                if env.get_wrapper_attr('was_real_done'):
                    assert 'episode' in info
                    
                    if 'episode' in info: # 'episode' is a default stats_key for RecordEpisodeStatistics
                        episode_stats = info['episode']
                        print(f'{step:05}', episode_stats)
                else:
                    assert not 'episode' in info
            else:
                assert not 'episode' in info

TypeError: Agent.Params.__init__() got an unexpected keyword argument 'actions_count'

## Configure

In [51]:
# @launchit.disable
# @launchit.collect_1
# ...
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': None,
 'env_id': 'BreakoutNoFrameskip-v4',
 'envs_count': 8,
 'gamma': None,
 'rollouts_count': None,
 'steps_count': None,
 'warmup_rollouts_count': None,
 'oracle_rollouts_period': None,
 'start_e': None,
 'end_e': None,
 'exploration_fraction': None,
 'replay_buffer_size': None,
 'tau': None,
 'capture_video': None,
 'batch_size': None,
 'optimizer': None,
 'learn_rate': None}


## Create

In [52]:
# @launchit.disable_2
ap = Agent.Params(
    actions_count=LS.envs.single_action_space.n.item()
)
LS.agent = Agent(ap).to(CONFIG.cuda_device)
LS.oracle = Agent(ap).to(CONFIG.cuda_device)
LS.oracle.load_state_dict(LS.agent.state_dict())

<All keys matched successfully>

# Video

## get_fresh_video_dir_name

In [19]:
def get_fresh_video_dir_name():
    lc = HP.launch_component()
    timestamp = datetime.datetime.now().strftime("%Y.%m.%d-%H:%M:%S") # generate unique dir name in order to shut up RecordVideo from complaining
    return os.path.join(CONFIG.run_path, f'video-{lc.name}-launch{lc.version}-{timestamp}')

## capture_video_of_test_rollout

In [55]:
def capture_video_of_test_rollout(agent, max_steps_count=10_000, video_dir_name=None):
    video_dir_name = lu.coalesce(video_dir_name, lambda: get_fresh_video_dir_name())
    assert video_dir_name is not None
    env = create_env(HP.env_id, video_dir_name=video_dir_name, is_auto_reset=True)
    assert env.action_space.n.item() == agent.params.actions_count
    obs, _ = env.reset(seed=HP.random_seed)
    device = next(iter(agent.parameters())).device
    
    with eval_guard(agent):
        with torch.no_grad():
            for step in range(0, max_steps_count): 
                obs = torch.tensor(einops.rearrange(obs, 'f h w -> 1 f h w')).to(device)
                logits = agent(obs)[0]
                action = torch.argmax(logits).cpu().item()
                obs, reward, terminated, truncated, info = env.step(action)
                
                if (terminated or truncated) and env.get_wrapper_attr('was_real_done'):
                    break

    game_meta = dict(
        reward=env.get_wrapper_attr('episode_returns'),
        frames_count=env.get_wrapper_attr('episode_lengths'),
        steps_count=step + 1,
    )
    
    env.close() # this forces video recording to complete and write video file
    
    video_fnames = list(filter(lambda fn: os.path.isfile(os.path.join(video_dir_name, fn)), os.listdir(video_dir_name)))
    assert len(video_fnames) == 1, len(video_fnames)
    video_fname = os.path.join(video_dir_name, video_fnames[0])
    video_meta = {}
    
    with open(video_fname, 'rb') as f:
        container = av.open(f)
        video_meta['fps'] = float(container.streams.video[0].average_rate)
        video_stream = container.streams.video[0]
        video_meta['duration'] = float(video_stream.duration * video_stream.time_base)
        
    return video_fname, dict(game=game_meta, video=video_meta)

In [56]:
# @launchit.disable
capture_video_of_test_rollout(LS.agent, max_steps_count=2000)

('/home/misha/dev/mine/neurolab/run/17_rl/video-17b_dqn_atari_01-launch0-2026.04.30-10:28:17.069524/rl-video-episode-0.mp4',
 {'game': {'reward': 0.0, 'frames_count': 524, 'steps_count': 119},
  'video': {'fps': 30.0, 'duration': 17.5}})

# TRAIN

## Configure

In [59]:
# @launchit.disable
# @launchit.collect

# RL params
HP.gamma = 0.99 # the discount factor gamma

# Training procedure params (DQN related) 
HP.global_steps_count = 100_000
HP.warmup_steps_count = 80_000
HP.learn_steps_period = 4 
HP.oracle_steps_period = 1_000
HP.start_e = 1 
HP.end_e = 0.01 
HP.exploration_fraction = 0.1 
HP.replay_buffer_size = 100_000
HP.tau = 1 
HP.capture_video = 'every(10000)'

# Optimization params
HP.batch_size = 32
HP.optimizer = 'Adam()'
HP.learn_rate = 1e-4

# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': None,
 'env_id': 'BreakoutNoFrameskip-v4',
 'envs_count': 8,
 'gamma': 0.99,
 'rollouts_count': 60000,
 'steps_count': 4,
 'warmup_rollouts_count': 1,
 'oracle_rollouts_period': 250,
 'start_e': 1,
 'end_e': 0.01,
 'exploration_fraction': 0.1,
 'replay_buffer_size': 100000,
 'tau': 1,
 'capture_video': 'every(10000)',
 'batch_size': 32,
 'optimizer': 'Adam()',
 'learn_rate': 0.0001}


## Create

In [58]:
ump = hp_parse_universal_module(HP.optimizer)
assert not ump.args
lr_params = hp_parse_learn_rate(HP.learn_rate)
optimizer = getattr(torch.optim, ump.module_name)(LS.agent.parameters(), lr=lr_params.learn_rate, **ump.kwargs)
capture_video_policy = create_capture_video_policy(HP.capture_video)

## Train

In [62]:
global_step = 0
start_time = time.time()
rb = ReplayBuffer(
    buffer_size=HP.replay_buffer_size * HP.envs_count, 
    observation_space=LS.envs.single_observation_space,
    action_space=LS.envs.single_action_space,
    device=CONFIG.cuda_device,
    optimize_memory_usage=True,
    handle_timeout_termination=False,
    n_envs=HP.envs_count,
)
obs, _ = LS.envs.reset(seed=HP.random_seed)
last_episode_rewards = np.zeros(HP.envs_count)
last_episode_lengths = np.zeros(HP.envs_count)

def linear_schedule(start_e: float, end_e: float, duration: int, t: int):
    slope = (end_e - start_e) / duration
    return max(slope * t + start_e, end_e)

for global_step in tqdm(range(0, HP.global_steps_count, HP.envs_count)):
    epsilon = linear_schedule(HP.start_e, HP.end_e, HP.exploration_fraction * HP.global_steps_count, global_step)
    
    if RNG.random() < epsilon:
        actions = np.array([LS.envs.single_action_space.sample() for _ in range(LS.envs.num_envs)])
    else:
        q_values = LS.agent(torch.Tensor(obs).to(CONFIG.cuda_device))
        actions = torch.argmax(q_values, dim=1).cpu().numpy()

    # TRY NOT TO MODIFY: execute the game and log data.
    # Note: truncation may happen due to max_num_frames_per_episode=108_000 frames limit in ALE
    # (30 minutes of real-time play, assuming the standard 60 frames per second)
    next_obs, rewards, terminations, truncations, infos = LS.envs.step(actions)

    # TRY NOT TO MODIFY: save data to reply buffer; handle `final_observation`
    real_next_obs = next_obs.copy()
    dones = np.logical_or(terminations, truncations)
    rb.add(obs, real_next_obs, actions, rewards, dones, infos)

    # TRY NOT TO MODIFY: CRUCIAL step easy to overlook
    obs = next_obs
    
    if 'episode' in infos: # 'episode' is a default stats_key for RecordEpisodeStatistics
        episode_stats = infos['episode']
        last_episode_rewards[episode_stats['_r']] = episode_stats['r'][episode_stats['_r']]
        last_episode_lengths[episode_stats['_l']] = episode_stats['l'][episode_stats['_l']]

    if global_step >= HP.warmup_steps_count:
        if (global_step % HP.learn_steps_period) == 0:
            data = rb.sample(HP.batch_size)
            
            with torch.no_grad():
                target_max, _ = LS.oracle(data.next_observations).max(dim=1)
                td_target = data.rewards.flatten() + HP.gamma * target_max * (1 - data.dones.flatten())
    
            # data.actions.shape = [batch_size,1], gather will pick from Agent's logits values by indices=data.actions,
            # while squeeze removes first dimension
            old_val = LS.agent(data.observations).gather(1, data.actions).squeeze()
    
            loss = F.mse_loss(old_val, td_target)
    
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # update Oracle weights
        if (global_step % HP.oracle_steps_period) == 0:
            for oracle_param, agent_param in zip(LS.oracle.parameters(), LS.agent.parameters()):
                oracle_param.data.copy_(HP.tau * agent_param.data + (1.0 - HP.tau) * oracle_param.data)
                
        summary_writer.add_scalar("losses/td_loss", loss.item(), global_step)
        summary_writer.add_scalar("losses/q_values", old_val.mean().item(), global_step)
        
    summary_writer.add_scalar("charts/sps", int(global_step / (time.time() - start_time)), global_step)
    summary_writer.add_scalar("charts/episodic_return", last_episode_rewards.mean(), global_step)
    summary_writer.add_scalar("charts/episodic_length", last_episode_lengths.mean(), global_step)

    if capture_video_policy(global_step):
        video_fname, video_meta = capture_video_of_test_rollout(LS.agent, video_dir_name=get_fresh_video_dir_name(), max_steps_count=5000)
        _, video_fname_ext = os.path.splitext(video_fname)
        ts = datetime.datetime.now().strftime('%Y.%m.%d-%H:%M:%S')
        remote_video_fname = f'{ts}-{global_step:09}.{video_fname_ext.lstrip('.')}'
        summary_writer.add_file(video_fname, remote_video_fname)
        summary_writer.add_file(io.StringIO(json.dumps(video_meta)), remote_video_fname + '.meta')
        ref_text = f'<a href="http://tensorboard-videos:6007/{summary_writer.log_dir}/{remote_video_fname}" target="_blank">{remote_video_fname}</a>'
        summary_writer.add_text('videos', ref_text, global_step)

    summary_writer.flush()

  0%|          | 0/60000 [00:00<?, ?it/s]

AssertionError: 

## Save

In [ ]:
# @launchit.disable_2
artifact_registry = new_artifact_registry()
lc = HP.launch_component()

with io.BytesIO() as b:
    torch.save(LS.agent.state_dict(), b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='pt', asset_classifier='agent', replace=True)

with io.StringIO() as b:
    json.dump(dataclasses.asdict(LS.agent.params), b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='agent_params', replace=True)

# LaunchIt!

## TRAIN

In [11]:
# @launchit.disable
launchit_t0 = time.time()

In [12]:
# @launchit.disable
launchit_interval = time.time() - launchit_t0

if launchit_interval > 0.05:
    lc = HP.launch_component()
    component_version = int(Autoincrement.get(lc.uri))
    assert component_version > 0, component_version
    artifact_registry_obj = new_artifact_registry(is_real=True)
    artifact_registry_obj.register_component(lc.name, component_version)
    LOG(f'Model instance registered, version={component_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_NAME=CONFIG.self_name,
        MODEL_VERSION=component_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN.value,
    )
    launch_notebook_fname = launchit.launchit(CONFIG.self_fname, launch_serial=component_version, expandvars=expandvars, collect_inds=[1], disable_inds=[1])
    LOG(f'Created launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=2
Creating /home/misha/dev/mine/neurolab/17_rl/17b_dqn_atari_02-launch2.ipynb
Created launch notebook "/home/misha/dev/mine/neurolab/17_rl/17b_dqn_atari_02-launch2.ipynb"


## Optuna (model selection)

### Templates

In [45]:
# @launchit.disable
# @launchit.collect_3
optuna_trial = optuna_multiprocessing.get_trial()

if optuna_trial is not None:
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    
    match study_serial:
        case 1:
            HP = Hyperparameters()
            HP.random_seed = 42
            assert False
        case _:
            assert False, f'Unsupported {study_serial=}'            

### Unleash

In [ ]:
# @launchit.disable
def get_optimize_directions(lg):
    match lg:
        case LaunchGoal.TRAIN_MODEL:
            return ['minimize']
        case _:
            assert False, f'Unsupported {lg=}'

lg = LaunchGoal.TRAIN_MODEL
expandvars = dict(
    PROJECT_ROOT_PATH=CONFIG.project_root_path,
    MODEL_GROUP_URI=LAUNCH_GOAL.model_group_uri,
    MODEL_NAME=LAUNCH_GOAL.model_name,
    LAUNCH_GOAL=lg.value,
)
study_serial = 1
study_name = f'{CONFIG.self_name}_{expandvars['LAUNCH_GOAL']}_{study_serial}'
rop_task = optuna_multiprocessing.RunOptimizationTask(
    app_name=CONFIG.self_name,
    is_stdout_enabled=False,
    notebook_fname=CONFIG.self_fname,
    notebook_name=CONFIG.self_name,
    model_group_uri=LAUNCH_GOAL.model_group_uri,
    model_name=LAUNCH_GOAL.model_name,
    expandvars=expandvars,
    collect_inds=[2],
    disable_inds=[],
    run_path=CONFIG.run_path,
    study_serial=study_serial,
    study_name=study_name,
    study_fname=os.path.join(CONFIG.run_path, study_name + '.log'),
    optimize_directions=get_optimize_directions(lg),
)
rop_tasks = [rop_task] * 1
mp_ctx = mp.get_context('spawn') # Req-d for CUDA, fork doesn't work within PyTorch

with mp_ctx.Pool(processes=4, maxtasksperchild=1) as pool:  # maxtasksperchild=1 forces fresh process for each trial to spare resources and avoid possible side effects of processe resue
    pool.map(optuna_multiprocessing.run_optimization, rop_tasks)

In [ ]:
# @launchit.disable
study = optuna.create_study(
    study_name=rop_task.study_name,
    storage=JournalStorage(JournalFileBackend(file_path=rop_task.study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs['MODEL_VERSION']}')
    
    LOG('  Params: ')
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    print(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        print(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        print(f"\tnumber: {trial.number}")
        print(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        print(f"\tparams: {trial.params}")
        print(f"\tvalues: {trial.values}")